In [4]:
import numpy as np
import scipy.io
import os, time
import matplotlib.pyplot as plt
from numba import njit
from statistics import mean, median

In [5]:
# ======================================================================
# 0. ADAPTIVE PGE SOLVERS FOR STANDARD NMF SUBPROBLEMS
# ======================================================================
# @njit(cache=True, fastmath=True)
def proj(x):
    return np.maximum(x, 0)

# @njit(cache=True, fastmath=True)
def compute_L_Lipschitz(A):
    """Frobenius-norm upper bound for the spectral norm of A.A^T."""
    return np.sqrt(np.sum(A*A)) # O(r^2) operations

# @njit(cache=True, fastmath=True)
def solve_AdaPGE_subproblem(A, C, B0, variant='AdaPGE1',
                            max_iter=200, tol=1e-6):
    """
    Solve  min_{B >= 0}  0.5*||A - BC||_F^2 
    using the adaptive projected-gradient exact-step method.
    C is the fixed factor, B is the variable factor.
    """
    m, n = A.shape
    # r = B.shape[1]
    B = B0.copy()

    # Precompute constants
    CCT = C@C.T
    ACT = A@C.T

    # L = compute_L_Lipschitz(CCT) 
    L=np.sqrt(np.sum(CCT*CCT)) # O(r^2) operations
    if variant != 'AdaPGE3':
        alpha_min = 0.5 / L
        alpha_max = 2.0 / L
    else:
        alpha_min = None
        alpha_max = None

    alpha_t = 1.0 / L
    if variant != 'AdaPGE1':
        gradB = (B @ CCT - ACT)
    BETA = 1e-4
    for t in range(max_iter):
        # Gradient of smooth part (without L2,0 penalty, lambda_val ignored here)
        if variant == 'AdaPGE1':
            gradB = (B @ CCT - ACT) 

        # Projected gradient stopping criterion
        if np.linalg.norm(proj(B - gradB) - B) < tol:
            break

        # Proximal point
        if variant in ['AdaPGE1', 'AdaPGE2']:
            Y = proj(B - (1.0 / L) * gradB)
        else:
            Y = proj(B - alpha_t*gradB)

        # Active-set identification (corrected rule)
        grad_Y = (Y @ CCT - ACT)
        mask_active = Y <= alpha_t * grad_Y
        mask_A1_plus = mask_active & (Y >= BETA)
        mask_free = ~mask_active

        # Direction P^t
        P = np.zeros_like(Y)
        P[mask_A1_plus] = -Y[mask_A1_plus]
        if np.any(mask_free):
            Z_candidate = proj(Y - alpha_t*grad_Y)
            P[mask_free] = Z_candidate[mask_free] - Y[mask_free]

        # Exact unconstrained step (Hessian = CCT + mu*I)
        a = np.sum(grad_Y*P)
        b = np.sum(P.T@P*CCT)
        eta_star = -a / b if b > 1e-12 else 1.0

        # Feasibility bound
        lambda_max = 1e20
        mask_neg = P < 0
        if np.any(mask_neg):
            ratios = -Y[mask_neg] / P[mask_neg]
            lambda_max = np.min(ratios)

        eta_star_star = min(eta_star, lambda_max)
        if eta_star_star < 0:
            eta_star_star = 0

        # Update B
        B_new = Y + eta_star_star * P

        # Curvature feedback for next step
        if variant == 'AdaPGE1':
            tau_next = eta_star_star
        else:  # AdaPGE2 and AdaPGE3
            grad_new = (B_new @ CCT - ACT)
            S = B_new - Y
            Y_diff = grad_new - grad_Y
            sy = np.sum(S * Y_diff)
            yy = np.sum(Y_diff * Y_diff)
            tau_next = sy / yy if yy > 1e-12 else 1.0

        # Adaptive step-size update (three cases)
        if variant == 'AdaPGE3':
            alpha_min = tau_next / 2.0
            alpha_max = 2.0 * tau_next

        if tau_next >= alpha_t:
            alpha_cand = min(2.0*alpha_t, (1.0 + 1.0 / tau_next)*alpha_t)
        elif alpha_t / 2.0 < tau_next < alpha_t:
            alpha_cand = tau_next
        else:
            alpha_cand = np.sqrt(alpha_t * tau_next / 2.0)

        alpha_next = max(alpha_min, min(alpha_max, alpha_cand))
        B = B_new
        alpha_t = alpha_next

    return B,t+1


In [ ]:

# @njit(cache=True, fastmath=True)
def compute_kkp_proj_norm(A, B, C):
    gradB = (B @ C - A) @ C.T
    gradC = B.T @ (B @ C - A)
    pgradB = gradB.copy()
    pgradC = gradC.copy()
    for i in range(B.shape[0]):
        for j in range(B.shape[1]):
            if B[i, j] == 0.0 and gradB[i, j] > 0.0:
                pgradB[i, j] = 0.0
    for i in range(C.shape[0]):
        for j in range(C.shape[1]):
            if C[i, j] == 0.0 and gradC[i, j] > 0.0:
                pgradC[i, j] = 0.0
    return np.sqrt(np.sum(pgradB**2) + np.sum(pgradC**2))

def rel_residual(A, B, C):
    return np.linalg.norm(A - B @ C, 'fro') / np.linalg.norm(A, 'fro')


def solve_NMF_ANLS(A, rank, solver_func, max_outer=50, inner_tol=1e-5,
                   inner_max=200, epsilonOUT=1e-8, variant='AdaPGE1', seed=0):
    rng = np.random.RandomState(seed)
    m, n = A.shape
    B = rng.rand(m, rank)
    C = rng.rand(rank, n)

    proj_norm0 = compute_kkp_proj_norm(A, B, C)

    if proj_norm0 < 1e-12: 
            return B, C, 0.0, 0.0, [0.0], [0.0], [rel_residual(A, B, C)], [0.0],[proj_norm0]


    obj_hist = [0.5*np.linalg.norm(A - B @ C, 'fro')**2]
    res_hist = [rel_residual(A, B, C)]
    time_hist = [0.0]
    proj_hist = [proj_norm0]
    inner_iters = []

    start_time = time.time()
    for _ in range(max_outer):
        # Update B (fix C)
        B,it1 = solver_func(A, C, B, variant=variant,
                                    max_iter=inner_max, tol=inner_tol)
        # Update C (fix B) – symmetric
        C,it2 = solver_func(A.T, B.T, C.T, variant=variant,
                                    max_iter=inner_max, tol=inner_tol)
        C=C.T
        # Track diagnostics
        time_hist.append(time.time() - start_time)

        obj = 0.5*np.linalg.norm(A - B @ C, 'fro')**2
        res = rel_residual(A, B, C)
        proj_norm = compute_kkp_proj_norm(A, B, C)
        obj_hist.append(obj)
        res_hist.append(res)
        proj_hist.append(proj_norm)
        inner_iters.append(it1+it2)
        if proj_norm<=epsilonOUT*proj_norm0:
            break

    return B, C, time_hist[-1], proj_hist[-1], inner_iters, obj_hist, res_hist, time_hist, proj_hist

def compute_condition_number(B, C, mu=1e-8):
    """
    Compute the maximum condition number of the Hessian matrices for the
    two NMF subproblems (B-block and C-block).
    Returns: max_cond, cond_B, cond_C
    """
    # Hessian for B-block: C^T C 
    H_B = C.T@C 
    eig_B = np.linalg.eigvalsh(H_B)
    # filter out near-zero eigenvalues (shouldn't happen due to mu)
    eig_B = eig_B[eig_B > 1e-12]
    cond_B = eig_B[-1]/eig_B[0] if len(eig_B) > 0 else np.inf

    # Hessian for C-block: B^T B 
    H_C = B.T@B 
    eig_C = np.linalg.eigvalsh(H_C)
    eig_C = eig_C[eig_C > 1e-12]
    cond_C = eig_C[-1]/eig_C[0] if len(eig_C) > 0 else np.inf

    max_cond = max(cond_B, cond_C)
    return max_cond

def compute_restric_kappa(A, SOL, TOL=1e-8):
    """
    Compute the condition number of the restricted Hessian for the factor SOL,
    given the other factor A.

    Parameters:
        A   : the fixed factor (e.g., C_star or B_star) of shape (m, r) or (n, r)
        SOL : the factor being examined (e.g., B_star or C_star) of shape (m, r) or (n, r)
        TOL : threshold to decide if a column is active (norm > TOL)

    Returns:
        kappa_restricted : condition number of the restricted Hessian, or 0/Inf if no active columns
    """
    # 1. Identify active columns of SOL (free variables)
    column_norms = np.linalg.norm(SOL, axis=0)
    free_indices = np.where(column_norms > TOL)[0]

    if len(free_indices) == 0:
        return 0.0   # no active columns -> no condition number

    # 2. Compute the full Hessian of the subproblem: A^T A  (size r x r)
    H_full = A.T @ A

    # 3. Restrict the Hessian to the free columns
    H_restricted = H_full[np.ix_(free_indices, free_indices)]

    # 4. Compute eigenvalues, filter out tiny ones, return condition number
    eigvals = np.linalg.eigvalsh(H_restricted)
    eigvals = eigvals[eigvals > 1e-12]
    if len(eigvals) == 0:
        return np.inf

    return eigvals[-1] / eigvals[0]


def compute_condition_number2(B_star, C_star):
    """
    Compute the maximum condition number of the Hessian matrices for the
    two NMF subproblems (B-block and C-block).
    Returns: q_BCD, max_cond
    """
    # Hessian for B-block: C^T C, restricted to active columns of B
    kappa_B = compute_restric_kappa(C_star, B_star)   # A = C_star, SOL = B_star

    # Hessian for C-block: B^T B, restricted to active columns of C
    kappa_C = compute_restric_kappa(B_star, C_star)   # A = B_star, SOL = C_star

    kappa = max(kappa_B, kappa_C)
    if kappa == np.inf or kappa <= 0:
        # If the problem is ill-conditioned, we return a conservative q = 1
        return 1.0, kappa

    # BCD contraction factor: q = ((sqrt(kappa)-1)/(sqrt(kappa)+1))^2
    # sqrt_kappa = np.sqrt(kappa)
    sqrt_kappa = kappa

    q = ((sqrt_kappa - 1) / (sqrt_kappa + 1)) ** 2
    return q, kappa

# ======================================================================
# 2. DATASET LOADING (real .mat files or synthetic fallback)
# ======================================================================

def load_dataset(name, data_dir='.'):
    ref_ranks = {
        'PBMC3K': 9, 'Swimmer': 16, 'AT&T Faces': 10, 'Web of Science': 11
        # 'Samson': 3, 'Jasper Ridge': 4, 'Urban': 4
    }
    file_map = {
        'PBMC3K': 'pbmc3k.mat', 'Swimmer': 'swimmer.mat', 'AT&T Faces': 'att_faces.mat',
        'Web of Science': 'wos.mat', 
        # 'Samson': 'samson.mat', 'Jasper Ridge': 'jasper_ridge.mat',
        # 'Urban': 'urban.mat'
    }
    fname = os.path.join(data_dir, file_map[name])
    if os.path.exists(fname):
        data = scipy.io.loadmat(fname)
        for key in ['data','X','V','M']:
            if key in data:
                X = data[key]
                break
        else:
            raise ValueError("No matrix found in " + fname)
    else:
        # Generate synthetic with the reference rank for demonstration
        r = ref_ranks[name]
        m = 200 if name in ['PBMC3K', 'Swimmer', 'AT&T Faces', 'Web of Science'] else 100
        n = 100 if name in ['PBMC3K', 'Web of Science'] else 500
        rng = np.random.RandomState(0)
        B = np.abs(rng.randn(m, r))
        C = np.abs(rng.randn(r, n))
        X = B@C + 0.05*np.abs(rng.randn(m, n))
        X = np.maximum(X, 0)
    return X, ref_ranks[name]

# ======================================================================
# NEW: Function to compute the theoretical BCD contraction factor
# ======================================================================


def compute_theoretical_BCD_rate(B, C, mu=1e-8):
    """
    Compute the effective contraction factor q_BCD for block coordinate descent.
    q_BCD = (((kappa) - 1) / ((kappa) + 1))^2
    Returns: q_BCD, kappa_max
    """
    # Hessian for B-block: C^T C + mu I
    H_B = C.T @ C 
    eig_B = np.linalg.eigvalsh(H_B)
    eig_B = eig_B[eig_B > 1e-12]
    cond_B = eig_B[-1] / eig_B[0] if len(eig_B) > 0 else np.inf

    # Hessian for C-block: B^T B + mu I
    H_C = B.T @ B 
    eig_C = np.linalg.eigvalsh(H_C)
    eig_C = eig_C[eig_C > 1e-12]
    cond_C = eig_C[-1] / eig_C[0] if len(eig_C) > 0 else np.inf

    kappa = max(cond_B, cond_C)
    if kappa == np.inf or kappa <= 0:
        return 1.0, kappa
    # sqrt_kappa = np.sqrt(kappa)
    q = ((kappa - 1) / (kappa + 1)) ** 2
    return q, kappa


if __name__ == "__main__":
    datasets = ['PBMC3K', 'Swimmer', 'AT&T Faces', 'Web of Science']
                # 'Samson', 'Jasper Ridge', 'Urban']
    methods = [
        ('AdaPGE1', 'AdaPGE1'),
        ('AdaPGE2', 'AdaPGE2'),
        ('AdaPGE3', 'AdaPGE3')
    ]
    NUM_TRIALS = 5
    MAX_OUTER = 50
    MIN_INNER = 1e-6
    MAX_INNER = 200

    for ds in datasets:
        X, ref_rank = load_dataset(ds)
        r = ref_rank
        print(f"\n=== Dataset: {ds} (shape {X.shape}, rank {r}) ===")

        histories = {name: {"obj": [], "time": [], "res": [], "final_time": [],
                            "final_res": [], "final_proj": [], "inner_iter": [],
                            "cond": [], "theo_res": [], "q": []}
                     for name, _ in methods}

        for trial in range(NUM_TRIALS):
            np.random.seed(trial * 100 + 1)
            for name, variant in methods:
                # Solve NMF
                U, V, total_time, fina_proj, inner_it, obj_hist, res_hist, time_hist, proj_hist = \
                    solve_NMF_ANLS(X, r, solve_AdaPGE_subproblem, max_outer=MAX_OUTER,
                                   inner_tol=MIN_INNER, inner_max=MAX_INNER,
                                   variant=variant, seed=trial*100+1)

                # compute kappa and q for theoretical BCD rate
                # q, max_cond=compute_condition_number2(U, V)

                # Compute condition number
                max_cond, _, _ = compute_condition_number(U, V, mu=1e-8)
                
                # Compute theoretical BCD rate
                q, kappa = compute_theoretical_BCD_rate(U, V, mu=1e-8)

                # Generate theoretical residual curve
                initial_res = res_hist[0]
                num_outer = len(res_hist)-1
                theo_res = [initial_res*(q**k) for k in range(num_outer+1)]

                # Store
                histories[name]["obj"].append(obj_hist)
                histories[name]["res"].append(res_hist)
                histories[name]["time"].append(time_hist)
                histories[name]["final_time"].append(total_time)
                histories[name]["final_proj"].append(fina_proj)
                histories[name]["inner_iter"].append(inner_it)
                histories[name]["final_res"].append(rel_residual(X, U, V))
                histories[name]["cond"].append(max_cond)
                histories[name]["theo_res"].append(theo_res)
                histories[name]["q"].append(q)

        # ---- Statistics Table ----
        print(f"{'Method':<12} | {'Time Mean':<10} | {'Time Med':<10} "
              f"| {'Obj Mean':<12} | {'Obj Med':<12} "
              f"| {'Res Mean':<10} | {'Res Med':<10} "
              f"| {'Proj Mean':<10} | {'Proj Med':<10} "
              f"| {'InIter Mean':<10} | {'InIter Med':<10} "
              f"| {'Cond Mean':<10} | {'Cond Med':<10} "
              f"| {'Theo Rate (q)':<15}")
        print("-" * 180)

        for name, _ in methods:
            t_mean = mean(histories[name]["final_time"])
            t_med = median(histories[name]["final_time"])
            objs = [h[-1] for h in histories[name]["obj"]]
            o_mean = mean(objs); o_med = median(objs)
            r_mean = mean(histories[name]["final_res"]); r_med = median(histories[name]["final_res"])
            pr_mean = mean(histories[name]["final_proj"]); pr_med = median(histories[name]["final_proj"])
            sum_iter = [sum(it) for it in histories[name]["inner_iter"]]
            it_mean = mean(sum_iter); it_med = median(sum_iter)
            cond_mean = mean(histories[name]["cond"]); cond_med = median(histories[name]["cond"])
            q_mean = mean(histories[name]["q"]); q_med = median(histories[name]["q"])

            print(f"{name:<12} | {t_mean:<10.4f} | {t_med:<10.4f} "
                  f"| {o_mean:<12.6f} | {o_med:<12.6f} "
                  f"| {r_mean:<10.6f} | {r_med:<10.6f} "
                  f"| {pr_mean:<10.6f} | {pr_med:<10.6f} "
                  f"| {it_mean:<10.2f} | {it_med:<10.2f} "
                  f"| {cond_mean:<10.2f} | {cond_med:<10.2f} "
                  f"| {q_mean:<15.6f}")
        print("=" * 180)

        # ---- Plot Empirical vs Theoretical Residual Curves ----
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        colors = {'AdaPGE1': 'blue', 'AdaPGE2': 'green', 'AdaPGE3': 'red'}
        for i, (name, _) in enumerate(methods):
            # Average the residual curves across trials
            max_len = max(len(res) for res in histories[name]["res"])
            padded_res = np.array([np.pad(res, (0, max_len - len(res)), constant_values=res[-1]) 
                                   for res in histories[name]["res"]])
            avg_res = np.mean(padded_res, axis=0)
            
            # Average the theoretical curves
            max_len_t = max(len(res) for res in histories[name]["theo_res"])
            padded_theo = np.array([np.pad(res, (0, max_len_t - len(res)), constant_values=res[-1]) 
                                    for res in histories[name]["theo_res"]])
            avg_theo = np.mean(padded_theo, axis=0)
            
            axes[i].semilogy(range(len(avg_res)), avg_res, 'o-', 
                             label='Empirical', color=colors[name], linewidth=2)
            axes[i].semilogy(range(len(avg_theo)), avg_theo, '--', 
                             label='Theoretical (q=%.4f)' % mean(histories[name]["q"]), 
                             color='black', linewidth=2)
            axes[i].set_xlabel('Outer Iterations')
            axes[i].set_ylabel('Relative Residual')
            axes[i].set_title(f'{name} on {ds}')
            axes[i].legend()
            axes[i].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig(f'convergence_{ds}.png', dpi=150)
        plt.close()
        print(f"Saved convergence plot for {ds}.\n")


/var/folders/m0/zls6mb711wjbmvvy6hv0w0jc0000gn/T/ipykernel_18739/3698063472.py:177: RuntimeWarning: divide by zero encountered in matmul
  X = B@C + 0.05*np.abs(rng.randn(m, n))
/var/folders/m0/zls6mb711wjbmvvy6hv0w0jc0000gn/T/ipykernel_18739/3698063472.py:177: RuntimeWarning: overflow encountered in matmul
  X = B@C + 0.05*np.abs(rng.randn(m, n))
/var/folders/m0/zls6mb711wjbmvvy6hv0w0jc0000gn/T/ipykernel_18739/3698063472.py:177: RuntimeWarning: invalid value encountered in matmul
  X = B@C + 0.05*np.abs(rng.randn(m, n))
/var/folders/m0/zls6mb711wjbmvvy6hv0w0jc0000gn/T/ipykernel_18739/3698063472.py:3: RuntimeWarning: divide by zero encountered in matmul
  gradB = (B @ C - A) @ C.T
/var/folders/m0/zls6mb711wjbmvvy6hv0w0jc0000gn/T/ipykernel_18739/3698063472.py:3: RuntimeWarning: overflow encountered in matmul
  gradB = (B @ C - A) @ C.T
/var/folders/m0/zls6mb711wjbmvvy6hv0w0jc0000gn/T/ipykernel_18739/3698063472.py:3: RuntimeWarning: invalid value encountered in matmul
  gradB = (B @ C - 


=== Dataset: PBMC3K (shape (200, 100), rank 9) ===


/var/folders/m0/zls6mb711wjbmvvy6hv0w0jc0000gn/T/ipykernel_18739/3698063472.py:71: RuntimeWarning: divide by zero encountered in matmul
  H_B = C.T@C
/var/folders/m0/zls6mb711wjbmvvy6hv0w0jc0000gn/T/ipykernel_18739/3698063472.py:71: RuntimeWarning: overflow encountered in matmul
  H_B = C.T@C
/var/folders/m0/zls6mb711wjbmvvy6hv0w0jc0000gn/T/ipykernel_18739/3698063472.py:71: RuntimeWarning: invalid value encountered in matmul
  H_B = C.T@C
/var/folders/m0/zls6mb711wjbmvvy6hv0w0jc0000gn/T/ipykernel_18739/3698063472.py:78: RuntimeWarning: divide by zero encountered in matmul
  H_C = B.T@B
/var/folders/m0/zls6mb711wjbmvvy6hv0w0jc0000gn/T/ipykernel_18739/3698063472.py:78: RuntimeWarning: overflow encountered in matmul
  H_C = B.T@B
/var/folders/m0/zls6mb711wjbmvvy6hv0w0jc0000gn/T/ipykernel_18739/3698063472.py:78: RuntimeWarning: invalid value encountered in matmul
  H_C = B.T@B
/var/folders/m0/zls6mb711wjbmvvy6hv0w0jc0000gn/T/ipykernel_18739/3698063472.py:199: RuntimeWarning: divide by zero

Method       | Time Mean  | Time Med   | Obj Mean     | Obj Med      | Res Mean   | Res Med    | Proj Mean  | Proj Med   | InIter Mean | InIter Med | Cond Mean  | Cond Med   | Theo Rate (q)  
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
AdaPGE1      | 1.5123     | 1.5564     | 35.846604    | 36.386116    | 0.010137   | 0.010232   | 5.637936   | 5.286245   | 16740.40   | 17095.00   | 89.82      | 74.66      | 0.952209       
AdaPGE2      | 2.0410     | 2.0365     | 8.436120     | 8.150964     | 0.004925   | 0.004843   | 3.253008   | 2.818676   | 20000.00   | 20000.00   | 70.88      | 68.36      | 0.942316       
AdaPGE3      | 2.0956     | 2.0799     | 216.613617   | 179.891845   | 0.022846   | 0.022752   | 178.171115 | 208.071613 | 20000.00   | 20000.00   | 169.35     | 153.06     | 0.976146       
Saved convergence plot for PBMC3K.


=== Dataset: Swim